Extract data from csv

In [86]:
import pandas as pd
import numpy as np
import datetime

In [ ]:
df = pd.read_csv('data/bronze/cafe_sales.csv')

In [ ]:
df.sample(5)

In [ ]:
df.info()

Transform data

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy.columns = [c.lower().replace(' ', '_') for c in df_copy.columns]

In [ ]:
df_copy.columns

In [ ]:
df_copy['item'].unique()

In [ ]:
def is_missing_flag(df, columns, value): 
    for column in columns:
        df.loc[(df[column] == value) | (df[column].isna()), 'is_missing'] = True

In [ ]:
def is_error_flag(df, columns, value): 
    for column in columns:
        df.loc[df_copy[column] == value, 'is_error'] = True

In [ ]:
def define_flags(df):
   df['is_error'] = False

   df['is_missing'] = False

   columns_for_missing_flag = ['item', 'quantity', 'price_per_unit', 'total_spent', 'transaction_date']
   is_missing_flag(df, columns_for_missing_flag, 'UNKNOWN')

   columns_for_error_flag = ['item', 'quantity', 'price_per_unit', 'total_spent']
   is_error_flag(df, columns_for_error_flag, 'ERROR')

   return df

In [ ]:
df_copy = define_flags(df_copy)

In [ ]:
df_copy[df_copy['is_missing'] == True]

In [ ]:
df_copy.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

In [ ]:
df_copy

In [ ]:
df_copy.info()

In [ ]:
df_copy['price_per_unit'] = df_copy['price_per_unit'].astype(float)

In [ ]:
df_copy['total_spent'] = df_copy['total_spent'].astype(float)

In [ ]:
df_copy['quantity'] = pd.to_numeric(df_copy['quantity'], errors='coerce').astype('Int64')

In [ ]:
df_copy['transaction_date'] = pd.to_datetime(df_copy['transaction_date'])

In [ ]:
type(df_copy['quantity'][0])

In [ ]:
df_copy.loc[
   (df_copy['price_per_unit'].isna() &
    df_copy['total_spent'].notna() &
    df_copy['quantity'].notna()
    ), 'price_per_unit'] = df_copy['total_spent']/df_copy['quantity']

In [ ]:
df_copy.info()

In [ ]:
df_copy['item'].unique()

In [ ]:
df_copy[df_copy['item'].isna()]

In [ ]:
mapping_df = df_copy.loc[
   (df_copy['item'].notna()) & 
   (df_copy['price_per_unit'].notna()),
   ['item', 'price_per_unit']].drop_duplicates()

In [ ]:
mapping_price_dict = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [ ]:
mapping_price_dict

In [ ]:
mapping_df.drop_duplicates(subset=['price_per_unit'], keep=False, inplace=True)

In [ ]:
mapping_item_dict = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [ ]:
mapping_item_dict

In [ ]:
for item, price in mapping_item_dict.items():
   mask = (df_copy['item'].isna()) & (df_copy['price_per_unit'] == price)
   df_copy.loc[mask, 'item'] = item

In [ ]:
df_copy[df_copy['item'].isna()]

In [ ]:
for item, price in mapping_price_dict.items():
   mask = (df_copy['price_per_unit'].isna()) & (df_copy['item'] == item)
   df_copy.loc[mask, 'price_per_unit'] = price

In [ ]:
df_copy.loc[
   (df_copy['price_per_unit'].notna()) &
   (df_copy['total_spent'].isna()) &
   (df_copy['quantity'].notna()
   ), 'total_spent'] = df_copy['price_per_unit'] * df_copy['quantity']

In [ ]:
df_copy.info()

In [ ]:
df_copy[df_copy['total_spent'].isna()]

Load into silver parquet

In [ ]:
df_copy.to_parquet('data/silver/cleaned_cafe_sales.parquet')

Create gold parquet

In [83]:
df_copy

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,False,False
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,False,False
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,True,False
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27,False,False
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,False,False
...,...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30,False,False
9996,TXN_9659401,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-06-02,False,True
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02,False,False
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-12-02,False,True


In [91]:
df_gold = df_copy[df_copy['is_error'] == False].copy()

In [92]:
df_gold

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,False,False
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,False,False
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27,False,False
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,False,False
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31,False,False
...,...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30,False,False
9996,TXN_9659401,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-06-02,False,True
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02,False,False
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-12-02,False,True


In [95]:
df_gold['transaction_month'] =  pd.to_datetime(df_gold['transaction_date']).dt.month_name()

In [110]:
df_gold['day_of_the_week'] = pd.to_datetime(df_gold['transaction_date']).dt.dayofweek

In [108]:
df_gold['day_of_the_week'] = pd.to_datetime(df_gold['transaction_date']).dt.day_name()

In [109]:
df_gold

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing,transaction_month,day_of_the_week
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,False,False,September,Friday
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,False,False,May,Tuesday
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27,False,False,April,Thursday
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,False,False,June,Sunday
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31,False,False,March,Friday
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30,False,False,August,Wednesday
9996,TXN_9659401,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-06-02,False,True,June,Friday
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02,False,False,March,Thursday
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-12-02,False,True,December,Saturday
